# 06 · Spark Partitioning & Performance

Este notebook contioene diferentes secciones en la que se ejecuta una carga de trabajo bajo distintas condiciones (número de particiones, `repartition` vs `coalesce`, con/sin `cache`, join normal vs broadcast, con/sin `partitionBy` en escritura) y registra tiempos y métricas.

Al final, todos los resultados se consolidan en `results/experiment_results.csv`, y las conclusiones se recogen en la sección final de este notebook.


## 1. Setup

Configuración de la sesión de Spark, igual que en los proyectos anteriores.

In [1]:
from pathlib import Path
import sys
import time
import statistics

PROJECT_ROOT = Path.cwd().resolve()

while not (PROJECT_ROOT / "common").exists():
    if PROJECT_ROOT.parent == PROJECT_ROOT:
        raise RuntimeError("No se encontró la carpeta 'common'.")
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from common.spark_session import create_spark_session
from pyspark.sql import functions as F

spark = create_spark_session(app_name="06-partitioning-performance")
spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/09/28 10:38:52 WARN Utils: Your hostname, MacBook-Air-de-Gloria.local, resolves to a loopback address: 127.0.0.1; using 192.168.1.20 instead (on interface en0)
26/09/28 10:38:52 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/28 10:39:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## 2. Carga de datos

Reutilizamos el dataset de NYC Taxi (`datasets/02-nyc-taxi`), el mismo del proyecto 02.

In [2]:
DATASETS_PATH = PROJECT_ROOT / "datasets" / "02-nyc-taxi"

for file in DATASETS_PATH.iterdir():
    print(file.name)

yellow_tripdata_2026-05.parquet


In [3]:
INPUT_PATH = DATASETS_PATH / "yellow_tripdata_2026-05.parquet"

# Lista de ficheros que vamos a analizar
parquet_files = [INPUT_PATH]

print("=== FICHEROS DE ENTRADA ===")

for file in parquet_files:

    # Tamaño físico del fichero
    size_mb = file.stat().st_size / (1024 ** 2)

    # Lectura individual del fichero con Spark
    df_file = spark.read.parquet(str(file))

    # Información del DataFrame generado
    n_rows = df_file.count()
    n_partitions = df_file.rdd.getNumPartitions()

    print(f"\nFichero: {file.name}")
    print(f"Tamaño:      {size_mb:.2f} MB")
    print(f"Registros:   {n_rows:,}")
    print(f"Particiones: {n_partitions}")

=== FICHEROS DE ENTRADA ===

Fichero: yellow_tripdata_2026-05.parquet
Tamaño:      66.47 MB
Registros:   4,090,836
Particiones: 8


In [4]:
df_file.printSchema()

root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)



## 3. Utilidad: medir tiempos

Spark es *lazy*: una transformación no ejecuta nada hasta que hay una acción (`.count()`, `.collect()`, `.write()`...). Medir el tiempo de una transformación sin acción solo mide cuánto tarda Spark en construir el plan, no la ejecución real.

Además, el tiempo que se observa nunca depende solo de la operación que queremos estudiar:

Tiempo observado = procesamiento + I/O + scheduling + inicialización/JIT + cachés + variabilidad del sistema

Dos causas concretas de que la primera ejecución sea más lenta que las siguientes, sin que eso signifique que la configuración "mejoró":
- la JVM usa JIT (Just-In-Time compilation), que va optimizando el código a medida que se ejecuta varias veces;
- la page cache del sistema operativo puede conservar en RAM bloques del fichero leídos la primera vez, así que una segunda lectura no siempre vuelve a tocar disco (esto es distinto de `df.cache()` / `df.persist()`, que es caché explícita de Spark).

Para este proyecto usaremos una metodología sencilla y consistente en todos los experimentos:

**1 ejecución de warm-up (se descarta) + 3 ejecuciones medidas → mediana de las tres.**

Usamos la mediana en vez de la media porque es menos sensible a una ejecución puntualmente anómala (por ejemplo, si el sistema estaba haciendo otra cosa en ese momento).

In [5]:
import time
import statistics

results = []

def time_it(
    action_fn,
    experiment,
    strategy,
    repeats=3,
    warmup=True,
    **metadata
):
    """
    Ejecuta una acción varias veces y registra sus tiempos.

    Parameters
    ----------
    action_fn : callable
        Acción que fuerza la ejecución del workload de Spark.

    experiment : str
        Nombre del experimento al que pertenece la medición.

    strategy : str
        Estrategia o configuración concreta evaluada.

    repeats : int
        Número de ejecuciones medidas.

    warmup : bool
        Si True, realiza una ejecución previa que no se contabiliza.

    **metadata
        Información adicional opcional sobre el experimento
        (número de particiones, tipo de join, etc.).

    Returns
    -------
    float
        Mediana de los tiempos medidos.
    """

    # Warm-up
    if warmup:
        action_fn()

    # Ejecuciones medidas
    times = []

    for _ in range(repeats):
        start = time.perf_counter()
        action_fn()
        elapsed = time.perf_counter() - start
        times.append(elapsed)

    median_time = statistics.median(times)

    # Registro del experimento
    result = {
        "experiment": experiment,
        "strategy": strategy,
        "runs": times,
        "median_s": median_time,
        **metadata
    }

    results.append(result)

    return median_time

## 4. Experimento 1 — Baseline

Antes de tocar nada, buscamos conocer el punto de partida:

- número de registros y de particiones actuales
- número y tamaño de los ficheros de entrada
- tiempo de una carga de trabajo de referencia, **tal cual viene el DataFrame**

El baseline es nuestro punto de referencia: medir cómo se comporta el dataset tal y como Spark lo ha cargado, antes de hacer repartition(), coalesce(), cache(), etc.

Para que la carga de trabajo de referencia sirva como instrumento de medida, debe cumplir cuatro características:  
  
**1. Que procese una cantidad relevante del dataset.**
No queremos algo que filtre el 99,9 % de los datos antes de empezar.
  
**2. Que provoque shuffle.**
Estamos estudiando particionado y rendimiento, así que queremos una operación sensible a cómo Spark distribuye los datos.
  
**3. Que sea suficientemente sencilla para entenderla.**
Con cinco joins, tres ventanas y veinte agregaciones, sería difícil saber qué está ocurriendo cuando cambie el rendimiento.
  
**4. Que sea exactamente la misma en todos los experimentos.**
Esta es fundamental.
  

**reference_workload() define qué trabajo va a hacer Spark, mientras que time_it() mide cuánto tarda Spark en hacerlo.**

Para la pregunta del `¿Cómo afecta el particionado al rendimiento de una carga de trabajo Spark?` necesitamos diseñar una carga de trabajo representativa y estable para usarla como instrumento de medida. Por eso, necesitamos una operación cuyo rendimiento pueda verse afectado por cómo están distribuidos los datos entre las particiones (con un groupBy) y añadimos algunas agregaciones sencillas, razonables para NYC Taxi y suficientemente representativas.

> Semánticamente estamos preguntando:
> Para cada zona de recogida, ¿cuántos viajes se han iniciado y cuál ha sido su tarifa y distancia media?

In [6]:
# ============================================================
# EXPERIMENTO 1 — BASELINE
# ============================================================

INPUT_PATH = DATASETS_PATH / "yellow_tripdata_2026-05.parquet"

# Lectura del fichero
df_file = spark.read.parquet(str(INPUT_PATH))

# Características del fichero y del DataFrame
size_mb = INPUT_PATH.stat().st_size / (1024 ** 2)
n_rows = df_file.count()
n_partitions = df_file.rdd.getNumPartitions()

print("=== FICHERO DE ENTRADA ===")
print(f"Fichero:      {INPUT_PATH.name}")
print(f"Tamaño:       {size_mb:.2f} MB")
print(f"Registros:    {n_rows:,}")
print(f"Particiones:  {n_partitions}")


# Carga de trabajo de referencia
def reference_workload(df):
    """
    Agrupa los viajes por zona de recogida y calcula:
    - número de viajes
    - tarifa media
    - distancia media

    Solo define transformaciones; no ejecuta ninguna action.
    """
    return (
        df.groupBy("PULocationID")
          .agg(
              F.count("*").alias("trip_count"),
              F.avg("fare_amount").alias("avg_fare"),
              F.avg("trip_distance").alias("avg_distance")
          )
    )


# Medición del baseline
baseline_time = time_it(
    lambda: reference_workload(df_file).collect(),
    experiment="baseline",
    strategy="initial_partitioning",
    partitions=df_file.rdd.getNumPartitions()
)

print("\n=== BASELINE ===")
print(f"Mediana: {baseline_time:.3f} s")

=== FICHERO DE ENTRADA ===
Fichero:      yellow_tripdata_2026-05.parquet
Tamaño:       66.47 MB
Registros:    4,090,836
Particiones:  8

=== BASELINE ===
Mediana: 0.224 s


In [7]:
print(
    "defaultParallelism:",
    spark.sparkContext.defaultParallelism
)

print(
    "maxPartitionBytes:",
    spark.conf.get("spark.sql.files.maxPartitionBytes")
)

print(
    "openCostInBytes:",
    spark.conf.get("spark.sql.files.openCostInBytes")
)

print(
    "minPartitionNum:",
    spark.conf.get("spark.sql.files.minPartitionNum", None)
)

print(
    "maxPartitionNum:",
    spark.conf.get("spark.sql.files.maxPartitionNum", None)
)

print(
    "shuffle.partitions:",
    spark.conf.get("spark.sql.shuffle.partitions")
)

print(
    "adaptive.enabled:",
    spark.conf.get("spark.sql.adaptive.enabled")
)

print(
    "adaptive.coalescePartitions.enabled:",
    spark.conf.get("spark.sql.adaptive.coalescePartitions.enabled")
)

defaultParallelism: 8
maxPartitionBytes: 134217728b
openCostInBytes: 4194304b
minPartitionNum: None
maxPartitionNum: None
shuffle.partitions: 4
adaptive.enabled: true
adaptive.coalescePartitions.enabled: true


In [8]:
# Distribución de registros por partición de lectura.
# Nota: las particiones vacías no aparecen en la salida,
# porque groupBy solo cuenta las filas que existen.

def partition_distribution(df):
    return (
        df
        .withColumn("_partition_id", F.spark_partition_id())
        .groupBy("_partition_id")
        .count()
        .orderBy("_partition_id")
    )


print("=== DISTRIBUCIÓN — PARTICIONES INICIALES ===")
partition_distribution(df_file).show()

=== DISTRIBUCIÓN — PARTICIONES INICIALES ===
+-------------+-------+
|_partition_id|  count|
+-------------+-------+
|            0|1048576|
|            2|1048576|
|            4|1048576|
|            6| 945108|
+-------------+-------+



<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### Conclusiones — Baseline

El dataset de entrada es **un único fichero Parquet de 66,47 MB y 4.090.836 registros**. Al cargarlo, Spark genera **8 particiones de lectura**, lo que demuestra que un fichero físico no equivale a una única partición de Spark.

Sin embargo, al inspeccionar el contenido de cada partición con `spark_partition_id()`, **solo 4 de las 8 contienen datos**:

| Partición | Registros |
|---:|---:|
| 0 | 1.048.576 |
| 1 | 0 |
| 2 | 1.048.576 |
| 3 | 0 |
| 4 | 1.048.576 |
| 5 | 0 |
| 6 | 945.108 |
| 7 | 0 |

(Las particiones vacías no aparecen en la salida de la celda porque `groupBy` solo cuenta filas existentes).

#### ¿Por qué 8 particiones?

La configuración de la sesión muestra:

- `defaultParallelism = 8`
- `spark.sql.files.maxPartitionBytes = 128 MB`
- `spark.sql.files.openCostInBytes = 4 MB`
- `spark.sql.files.minPartitionNum = None`
- `spark.sql.files.maxPartitionNum = None`

Spark no crea una partición por fichero: divide los ficheros en trozos (*splits*) de un tamaño máximo que calcula así:

```
maxSplitBytes = min(maxPartitionBytes, max(openCostInBytes, bytesPerCore))

bytesPerCore  = (66,47 MB + 4 MB de openCost) / 8 cores ≈ 8,8 MB
maxSplitBytes = min(128 MB, max(4 MB, 8,8 MB)) = 8,8 MB

66,47 MB / 8,8 MB ≈ 7,6  →  8 splits  →  8 particiones
```

Por eso `maxPartitionBytes` (128 MB) no es el límite que actúa aquí: con un fichero pequeño, Spark reduce el tamaño de los splits para repartir la lectura entre los 8 cores disponibles.

#### ¿Por qué solo 4 contienen datos?

Un fichero Parquet está organizado internamente en bloques llamados **row groups**, y un row group **no se puede dividir**: se lee entero dentro de una única partición. Spark asigna cada row group al split en el que cae su punto medio.

Este fichero tiene **4 row groups**: tres de **1.048.576 filas** (2²⁰, el tamaño por defecto de `pyarrow`, la librería habitual para generar estos ficheros) y un último con las **945.108** restantes. Cada row group ocupa aproximadamente dos splits, pero solo uno de ellos se queda con los datos:

```
Fichero (66 MB):   [ RG1 ~16 MB ][ RG2 ~16 MB ][ RG3 ~16 MB ][ RG4 ~16 MB ]
Particiones:        [P0] [P1]     [P2] [P3]     [P4] [P5]     [P6] [P7]
                     ✔    ✗        ✔    ✗        ✔    ✗        ✔    ✗
```

El experimento 6 lo confirma de forma independiente: al escribir sin `partitionBy` (sin shuffle, cada partición escribe su propio fichero) solo se generan `part-00000`, `part-00002`, `part-00004` y `part-00006`. Las particiones impares, vacías, no escriben nada.

#### Implicaciones

- El **paralelismo efectivo del baseline es 4, no 8**: cuatro tareas procesan ~1 millón de filas cada una y las otras cuatro terminan sin trabajo.
- El número de particiones que devuelve `getNumPartitions()` **no describe cómo se reparte realmente el trabajo**; hay que inspeccionar su contenido.
- Esta estructura explica también el resultado de `coalesce(4)` en el experimento 3.

#### Tiempo de referencia

Con este particionado inicial, la carga de trabajo de referencia obtiene una **mediana de 0,224 s**. La ejecución de warm-up, descartada, tardó 0,660 s: un ejemplo directo de por qué la primera ejecución no es representativa.

> **Baseline:** 1 fichero · 66,47 MB · 4.090.836 registros · 8 particiones de lectura (4 con datos) · **0,224 s**
</div>

## 5. Experimento 2 — Número de particiones

Se ejecuta la misma `reference_workload` variando el número de particiones antes de la agregación (1, 4, 8, 50 y 200) y se mide el tiempo en cada caso.

In [9]:
# ============================================================
# EXPERIMENTO 2 — NÚMERO DE PARTICIONES
# ============================================================

partition_values = [1, 4, 8, 50, 200]

for n in partition_values:

    # Definimos el nuevo particionado
    df_repartitioned = df_file.repartition(n)
    # Número de particiones solicitado/obtenido
    actual_partitions = df_repartitioned.rdd.getNumPartitions()

    # Medimos repartition + workload
    median_time = time_it(
        lambda df=df_repartitioned: reference_workload(df).collect(),
        experiment="partition_count",
        strategy="repartition",
        partitions=actual_partitions
    )

    print(
        f"Particiones solicitadas: {n:>3} | "
        f"Particiones reales: {actual_partitions:>3} | "
        f"Mediana: {median_time:.3f} s"
    )

Particiones solicitadas:   1 | Particiones reales:   1 | Mediana: 1.109 s


Particiones solicitadas:   4 | Particiones reales:   4 | Mediana: 1.510 s


Particiones solicitadas:   8 | Particiones reales:   8 | Mediana: 1.408 s


Particiones solicitadas:  50 | Particiones reales:  50 | Mediana: 1.776 s


Particiones solicitadas: 200 | Particiones reales: 200 | Mediana: 2.830 s


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### Conclusiones — Número de particiones

El baseline, con las **8 particiones iniciales generadas por Spark**, obtuvo una mediana de **0,224 s**.

Al aplicar `repartition(n)` antes de la carga de trabajo, todos los tiempos aumentaron:  
(los resultados de esta tabla podrán variar al reejecutar)

| Configuración | Particiones | Mediana |
|---|---:|---:|
| Baseline | 8 (4 con datos) | **0,224 s** |
| repartition(1) | 1 | 1,109 s |
| repartition(4) | 4 | 1,510 s |
| repartition(8) | 8 | 1,408 s |
| repartition(50) | 50 | 1,776 s |
| repartition(200) | 200 | 2,830 s |

Incluso `repartition(8)`, que produce el mismo número de particiones que el baseline (y además 8 particiones con datos, frente a las 4 útiles del baseline), es considerablemente más lento. La razón es que repartition() introduce una **redistribución de los datos mediante shuffle**, mientras que el baseline utiliza directamente las particiones de lectura ya existentes. Para este volumen de datos, el coste del shuffle supera cualquier beneficio de repartir mejor el trabajo.

Entre 1, 4 y 8 particiones las diferencias son pequeñas y no siguen un orden claro (`repartition(1)` fue el más rápido de los tres). En ejecuciones tan cortas y en local, estas diferencias entran dentro de la variabilidad de la medición. La tendencia clara aparece a partir de 50 y, sobre todo, de 200 particiones.

Los resultados también muestran que **aumentar el número de particiones no implica necesariamente mejorar el rendimiento**. Un mayor número de particiones puede aumentar el paralelismo, pero también genera más tareas, más trabajo de planificación y mayor overhead.

Este efecto es especialmente visible con `repartition(200)`: para un dataset de solo **66,47 MB** y un entorno con `defaultParallelism = 8`, crear 200 particiones genera muchas tareas pequeñas cuyo coste de gestión supera cualquier posible beneficio de paralelismo.

> **Conclusión:** el número de particiones debe ajustarse al volumen de datos, al workload y a los recursos disponibles. `repartition()` tiene un coste propio debido al shuffle y no debe utilizarse de forma indiscriminada.

### ⚠️ Particiones antes y después del shuffle

`repartition(n)` solo cambia el número de particiones **antes** de la agregación. El `groupBy` de `reference_workload` provoca su propio shuffle, y el número de particiones **después** de ese shuffle no lo decide `n`, sino `spark.sql.shuffle.partitions`. Su valor por defecto es 200, pero **en esta sesión está configurado a 4**, como muestra la celda de configuración del baseline (y el `Exchange hashpartitioning(payment_type, 4)` del experimento 5). Además, con **Adaptive Query Execution (AQE)** activado, Spark puede fusionar en tiempo de ejecución las particiones de shuffle pequeñas (`spark.sql.adaptive.coalescePartitions.enabled`).

Por tanto, este experimento mide el efecto del particionado de entrada y del shuffle adicional que introduce `repartition()`, pero no el del número de particiones de shuffle, que en agregaciones y joins suele ser el parámetro más determinante. En esta sesión, AQE está activado (`spark.sql.adaptive.enabled = true`) y también la fusión de particiones de shuffle (`coalescePartitions.enabled = true`).

### ¿De qué depende un buen número de particiones?

No existe un número de particiones óptimo de forma universal. El particionado adecuado depende principalmente de tres factores:

- **Volumen de datos:** al aumentar el volumen, generalmente es necesario aumentar también el número de particiones para repartir el trabajo y evitar que cada tarea procese cantidades excesivas de datos. Sin embargo, un número demasiado elevado genera particiones pequeñas y aumenta el overhead asociado a la creación, planificación y ejecución de tareas.

- **Workload:** el tipo de operaciones realizadas sobre los datos también condiciona el particionado. Operaciones como `groupBy`, `join`, `distinct` u `orderBy` pueden provocar **shuffles**, redistribuyendo los datos entre particiones. El número y la distribución de las particiones pueden afectar considerablemente al coste de estas operaciones. También debe considerarse la distribución de los datos, ya que un reparto desigual de determinadas claves puede provocar **data skew**.

- **Recursos disponibles:** el número de cores y executors determina cuántas tareas pueden ejecutarse en paralelo. Tener demasiado pocas particiones puede impedir aprovechar todo el paralelismo disponible, mientras que tener demasiadas genera un gran número de tareas pequeñas y aumenta el overhead de scheduling y ejecución.

Por tanto, existe un compromiso entre **tamaño de las particiones, paralelismo y overhead**:

- `pocas particiones → tareas grandes y menor paralelismo`

- `demasiadas particiones → tareas pequeñas y mayor overhead`

- `particionado adecuado → suficiente paralelismo sin generar tareas innecesariamente pequeñas`

No existe una fórmula exacta que determine el número óptimo para cualquier situación. Se utilizan criterios orientativos basados en el volumen de datos y los recursos del cluster, pero el ajuste final requiere observar el comportamiento real del workload mediante métricas, planes de ejecución y benchmarks.
</div>

## 6. Experimento 3 — `repartition` vs `coalesce`

Partiendo del DataFrame base, se reduce el número de particiones con ambas operaciones hasta el mismo número final (4).

In [10]:
# ============================================================
# EXPERIMENTO 3 — REPARTITION VS COALESCE
# ============================================================

n = 4

df_repartition = df_file.repartition(n)
df_coalesce = df_file.coalesce(n)

print("=== NÚMERO DE PARTICIONES ===")
print(f"Original:    {df_file.rdd.getNumPartitions()}")
print(f"Repartition: {df_repartition.rdd.getNumPartitions()}")
print(f"Coalesce:    {df_coalesce.rdd.getNumPartitions()}")


# ------------------------------------------------------------
# Planes de ejecución
# ------------------------------------------------------------

print("\n=== PLAN — REPARTITION ===")
df_repartition.explain()

print("\n=== PLAN — COALESCE ===")
df_coalesce.explain()


# ------------------------------------------------------------
# Benchmark
# ------------------------------------------------------------

repartition_time = time_it(
    lambda: reference_workload(df_repartition).collect(),
    experiment="repartition_vs_coalesce",
    strategy="repartition",
    partitions=4
)

coalesce_time = time_it(
    lambda: reference_workload(df_coalesce).collect(),
    experiment="repartition_vs_coalesce",
    strategy="coalesce",
    partitions=4
)

print("\n=== RESULTADOS ===")
print(f"repartition(4): {repartition_time:.3f} s")
print(f"coalesce(4):    {coalesce_time:.3f} s")


=== NÚMERO DE PARTICIONES ===
Original:    8


Repartition: 4
Coalesce:    4

=== PLAN — REPARTITION ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=true
+- == Final Plan ==
   ResultQueryStage 1
   +- ShuffleQueryStage 0
      +- Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM, [plan_id=1700]
         +- *(1) ColumnarToRow
            +- FileScan parquet [VendorID#44,tpep_pickup_datetime#45,tpep_dropoff_datetime#46,passenger_count#47L,trip_distance#48,RatecodeID#49L,store_and_fwd_flag#50,PULocationID#51,DOLocationID#52,payment_type#53L,fare_amount#54,extra#55,mta_tax#56,tip_amount#57,tolls_amount#58,improvement_surcharge#59,total_amount#60,congestion_surcharge#61,Airport_fee#62,cbd_congestion_fee#63] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(1 paths)[file:/Users/gloriadelriomarquez/Documents/Career/spark-lab/datasets/02..., PartitionFilters: [], PushedFilters: [], ReadSchema: struct<VendorID:int,tpep_pickup_datetime:timestamp_ntz,tpep_dropoff_datetime:timestamp_ntz,passen...
+- == In


=== RESULTADOS ===
repartition(4): 1.226 s
coalesce(4):    0.160 s


In [11]:
print("=== DISTRIBUCIÓN — REPARTITION(4) ===")
partition_distribution(df_repartition).show()

print("=== DISTRIBUCIÓN — COALESCE(4) ===")
partition_distribution(df_coalesce).show()

=== DISTRIBUCIÓN — REPARTITION(4) ===
+-------------+-------+
|_partition_id|  count|
+-------------+-------+
|            0|1022709|
|            1|1022709|
|            2|1022709|
|            3|1022709|
+-------------+-------+

=== DISTRIBUCIÓN — COALESCE(4) ===
+-------------+-------+
|_partition_id|  count|
+-------------+-------+
|            0|1048576|
|            1|1048576|
|            2|1048576|
|            3| 945108|
+-------------+-------+



<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### 🔀 Conclusiones — repartition() vs coalesce()

Partiendo de las **8 particiones iniciales**, se compararon dos estrategias para reducir el DataFrame a **4 particiones**:

| Estrategia | Particiones finales | Shuffle adicional | Mediana |
|---|---:|:---:|---:|
| repartition(4) | 4 | Sí | **1,226 s** |
| coalesce(4) | 4 | No | **0,160 s** |

Aunque ambas operaciones producen un DataFrame con **4 particiones**, el mecanismo utilizado por Spark es diferente.

#### 🔄 repartition(4)

El plan físico contiene **Exchange RoundRobinPartitioning(4), REPARTITION_BY_NUM**.

La presencia de **Exchange** indica que Spark realiza un **shuffle** para redistribuir los registros entre las cuatro nuevas particiones.

Esta redistribución permite obtener un reparto aproximadamente uniforme de los datos, pero introduce un coste adicional de ejecución.

#### 🪶 coalesce(4)

El plan físico contiene **Coalesce 4** y no aparece ningún **Exchange** asociado a esta operación.

Spark reduce el número de particiones aprovechando las particiones existentes, **sin realizar una redistribución global de los registros**.

Esto explica la importante diferencia observada en los tiempos: coalesce(4) es unas **7,7 veces más rápido** que repartition(4) para reducir el número de particiones.

#### 📊 Distribución resultante

| Partición | repartition(4) | coalesce(4) |
|---:|---:|---:|
| 0 | 1.022.709 | 1.048.576 |
| 1 | 1.022.709 | 1.048.576 |
| 2 | 1.022.709 | 1.048.576 |
| 3 | 1.022.709 | 945.108 |

`repartition(4)` reparte los registros de forma prácticamente exacta (round robin).

`coalesce(4)` junta particiones consecutivas: (0+1), (2+3), (4+5) y (6+7). Como en el baseline las particiones impares estaban vacías, cada partición resultante es **exactamente uno de los row groups originales**. El reparto sale equilibrado, pero por casualidad: depende por completo de cómo estaban distribuidos los datos de entrada.

#### ⚖️ El trade-off

Evitar el shuffle tiene una contrapartida: coalesce() **no realiza un rebalanceo global de los datos**, por lo que puede mantener particiones desiguales si las particiones originales estaban desbalanceadas.

repartition(), aunque es más costoso, **redistribuye los registros** y puede conseguir un reparto más equilibrado.

**Regla práctica:**

➡️ **Reducir particiones sin necesidad de redistribuir los datos → coalesce()**

➡️ **Aumentar particiones o redistribuir/rebalancear los datos → repartition()**

> 💡 El tiempo de coalesce(4), **0,160 s**, es algo inferior al baseline, **0,224 s**. Tiene sentido: ambos procesan los mismos 4 bloques de datos, pero coalesce(4) evita programar las 4 tareas vacías del baseline. Aun así, la diferencia absoluta es pequeña y las ejecuciones son muy cortas, por lo que este resultado no permite concluir que 4 particiones sean el número óptimo.

</div>

## 7. Experimento 4 — cache / persist

Se aplica una transformación no trivial sobre el DataFrame y se ejecutan varias acciones distintas sobre ella: primero sin persistencia y después con `.cache()`.

**Metodología.** Las tres acciones se agrupan en una misma función y se miden con `time_it` en los dos escenarios (warm-up + 3 ejecuciones → mediana). La excepción es la **materialización de la cache**: solo ocurre la primera vez que se ejecuta una acción tras `cache()`, así que no puede repetirse ni ir precedida de un warm-up (el warm-up sería precisamente la materialización). Por eso se mide una única vez, por separado, y se registra también en `results`.

In [12]:
# ============================================================
# EXPERIMENTO 4 — CACHE / PERSISTENCIA
# ============================================================

# Transformación que reutilizaremos varias veces
transformed = (
    df_file
    .filter(
        (F.col("trip_distance") > 0) &
        (F.col("fare_amount") > 0)
    )
    .withColumn(
        "fare_per_mile",
        F.col("fare_amount") / F.col("trip_distance")
    )
)


def cache_actions(df):
    """
    Mismas tres acciones en ambos escenarios (con y sin cache).
    """
    df.count()

    df.agg(
        F.avg("fare_amount").alias("avg_fare")
    ).collect()

    df.groupBy("payment_type").agg(
        F.avg("fare_per_mile").alias("avg_fare_per_mile")
    ).collect()


# ------------------------------------------------------------
# SIN CACHE — warm-up + 3 ejecuciones → mediana
# ------------------------------------------------------------

without_cache = time_it(
    lambda: cache_actions(transformed),
    experiment="cache",
    strategy="without_cache"
)


# ------------------------------------------------------------
# CON CACHE
# ------------------------------------------------------------

transformed.cache()

# Materialización: solo ocurre una vez, se mide fuera de time_it
start = time.perf_counter()
transformed.count()
cache_materialization = time.perf_counter() - start

results.append({
    "experiment": "cache",
    "strategy": "cache_materialization",
    "runs": [cache_materialization],
    "median_s": cache_materialization
})

# Reutilización: la cache ya está materializada,
# así que el warm-up no altera la medición
with_cache = time_it(
    lambda: cache_actions(transformed),
    experiment="cache",
    strategy="with_cache_reuse"
)


# Liberamos la cache
transformed.unpersist()


# ------------------------------------------------------------
# RESULTADOS
# ------------------------------------------------------------

print("=== CACHE / PERSISTENCIA ===")
print(f"Sin cache (mediana):               {without_cache:.3f} s")
print(f"Materialización cache:             {cache_materialization:.3f} s")
print(f"Reutilización con cache (mediana): {with_cache:.3f} s")
print(f"Cache total:                       {cache_materialization + with_cache:.3f} s")


26/09/28 10:40:14 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:14 WARN BlockManager: Persisting block rdd_520_0 to disk instead.
26/09/28 10:40:16 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:16 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:16 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:16 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:17 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:17 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:17 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)
26/09/28 10:40:17 WARN MemoryStore: Not

=== CACHE / PERSISTENCIA ===
Sin cache (mediana):               0.306 s
Materialización cache:             3.976 s
Reutilización con cache (mediana): 0.517 s
Cache total:                       4.493 s


26/09/28 10:40:19 WARN MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)


In [13]:
print("=== CONFIGURACIÓN DE MEMORIA ===")

spark_conf = spark.sparkContext.getConf()

driver_memory = spark_conf.get(
    "spark.driver.memory",
    "no configurado explícitamente"
)

executor_memory = spark_conf.get(
    "spark.executor.memory",
    "no configurado explícitamente"
)

memory_fraction = spark_conf.get(
    "spark.memory.fraction",
    "valor por defecto"
)

storage_fraction = spark_conf.get(
    "spark.memory.storageFraction",
    "valor por defecto"
)

print(f"Driver memory:    {driver_memory}")
print(f"Executor memory:  {executor_memory}")
print(f"Memory fraction:  {memory_fraction}")
print(f"Storage fraction: {storage_fraction}")

print("\n=== STORAGE LEVEL ===")

transformed.cache()

print(transformed.storageLevel)

transformed.unpersist()

=== CONFIGURACIÓN DE MEMORIA ===
Driver memory:    no configurado explícitamente
Executor memory:  no configurado explícitamente
Memory fraction:  valor por defecto
Storage fraction: valor por defecto

=== STORAGE LEVEL ===
Disk Memory Deserialized 1x Replicated


DataFrame[VendorID: int, tpep_pickup_datetime: timestamp_ntz, tpep_dropoff_datetime: timestamp_ntz, passenger_count: bigint, trip_distance: double, RatecodeID: bigint, store_and_fwd_flag: string, PULocationID: int, DOLocationID: int, payment_type: bigint, fare_amount: double, extra: double, mta_tax: double, tip_amount: double, tolls_amount: double, improvement_surcharge: double, total_amount: double, congestion_surcharge: double, Airport_fee: double, cbd_congestion_fee: double, fare_per_mile: double]

<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### 💾 Conclusiones — Cache y persistencia

Se ejecutaron las mismas tres acciones (`count`, una agregación global y un `groupBy`) sobre un mismo DataFrame transformado, primero **sin cache** y después utilizando **cache()**.

| Escenario | Tiempo |
|---|---:|
| Sin cache (mediana) | **0,306 s** |
| Materialización de la cache (1 ejecución) | **3,976 s** |
| Reutilización con cache (mediana) | **0,517 s** |
| Coste total con cache | **4,493 s** |

En este experimento la cache **empeoró el rendimiento en todos los sentidos**: la materialización fue costosa y, además, reutilizar el DataFrame cacheado fue **más lento** que recalcularlo desde el Parquet.

#### 🧠 La cache no es gratuita

La primera acción ejecutada después de cache() debe calcular el DataFrame completo y almacenarlo. En este experimento ese proceso necesitó **3,976 s**, más de diez veces lo que cuestan las tres acciones sin cache.

#### ⚠️ Presión de memoria

Durante la materialización Spark mostró avisos indicando que uno de los bloques no cabía en memoria:

**MemoryStore: Not enough space to cache rdd_520_0 in memory! (computed 73.7 MiB so far)**

**BlockManager: Persisting block rdd_520_0 to disk instead.**

`rdd_520_0` corresponde a la partición 0 del DataFrame cacheado. El StorageLevel utilizado por cache() es **Disk Memory Deserialized 1x Replicated** (equivalente a `MEMORY_AND_DISK`), así que Spark pudo guardar ese bloque en disco en lugar de descartarlo. La memoria del driver y del executor no está configurada explícitamente, por lo que la sesión trabaja con los valores por defecto.

El aviso **se repite durante la reutilización**: cada vez que una acción lee ese bloque desde disco, Spark intenta volver a subirlo a memoria y vuelve a fallar. Parte de cada lectura "cacheada" es, en realidad, una lectura de disco con un intento fallido de promoción a memoria.

El tamaño del fichero Parquet de entrada (**66,47 MB**) no representa la memoria necesaria para almacenar el DataFrame. Parquet utiliza compresión y codificación columnar, mientras que la representación cacheada es mucho más voluminosa.

#### 📉 Por qué recalcular salió más barato

Sin cache, cada acción vuelve a leer el Parquet, pero esa lectura es muy eficiente:

- Parquet es columnar y Spark solo lee **las columnas que necesita cada acción** (column pruning): `trip_distance`, `fare_amount` y `payment_type`, no las 21 columnas.
- Tras la ejecución de warm-up, el sistema operativo mantiene el fichero en su page cache, así que la lectura apenas toca disco.

La cache, en cambio, almacena **todas las columnas** del DataFrame transformado, y parte de ella acabó en disco. Para esta transformación sencilla, recalcular desde Parquet es más barato que leer la cache.

#### ♻️ ¿Cuándo compensa utilizar cache?

La cache resulta especialmente útil cuando un mismo DataFrame intermedio es **costoso de calcular y va a reutilizarse muchas veces**: por ejemplo, tras joins, agregaciones pesadas o lecturas de fuentes lentas.

Existe, por tanto, un coste que debe amortizarse:

**coste de materialización + coste de las reutilizaciones** frente a **coste de recomputar el lineage en cada acción**

En este caso no se cumple ninguna de las dos condiciones: la transformación es barata de recalcular (un filtro y una columna derivada sobre Parquet) y el DataFrame no cabe cómodamente en la memoria disponible.

> 💡 **Conclusión:** cache() no debe utilizarse automáticamente como una optimización. Su utilidad depende del coste de recomputación, del número de reutilizaciones y de la memoria disponible. En este experimento la materialización fue costosa, el DataFrame no cabía en memoria y la reutilización resultó más lenta que recalcular desde Parquet: la cache empeoró el rendimiento total.

</div>

<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### 💾 `cache()` vs `persist()`

En el experimento se ha utilizado **cache()** para almacenar temporalmente el DataFrame y evitar su recomputación en reutilizaciones posteriores.

Spark también permite utilizar **persist()**, que persigue el mismo objetivo pero permite seleccionar explícitamente el **StorageLevel**, es decir, la estrategia utilizada para almacenar las particiones del DataFrame.
  
---
```python
from pyspark import StorageLevel

# Forma cómoda: utiliza el StorageLevel por defecto de cache()
df_cached = transformed.cache()

# Persistencia indicando explícitamente el StorageLevel
df_memory = transformed.persist(StorageLevel.MEMORY_ONLY)
df_memory_disk = transformed.persist(StorageLevel.MEMORY_AND_DISK)
df_disk = transformed.persist(StorageLevel.DISK_ONLY)
```
---
    
| Estrategia | Comportamiento |
|---|---|
| **cache()** | Utiliza el StorageLevel por defecto de Spark para el DataFrame |
| **MEMORY_ONLY** | Mantiene las particiones en memoria; las que no puedan almacenarse deberán recomputarse cuando vuelvan a necesitarse |
| **MEMORY_AND_DISK** | Intenta almacenar las particiones en memoria y utiliza disco cuando no dispone de espacio suficiente |
| **DISK_ONLY** | Almacena las particiones en disco |

Por tanto:

- **cache()** → opción cómoda cuando el StorageLevel por defecto es adecuado.

- **persist(StorageLevel...)** → permite controlar explícitamente la estrategia de almacenamiento según las características del workload y los recursos disponibles.

En ambos casos, los datos persistidos pueden liberarse mediante: `transformed.unpersist()`

> 💡 **Idea clave:** `cache()` y `persist()` buscan evitar recomputaciones. La principal diferencia es que `persist()` permite seleccionar explícitamente el nivel de almacenamiento. Elegir un StorageLevel diferente implica un trade-off entre **memoria disponible, acceso a disco y coste de recomputación**.

</div>

## 8. Experimento 5 — Join convencional vs Broadcast Join

In [14]:
# ============================================================
# EXPERIMENTO 5 — JOIN CONVENCIONAL VS BROADCAST JOIN
# ============================================================

from pyspark.sql import functions as F

# Limpiamos posibles datos persistidos de experimentos anteriores
spark.catalog.clearCache()


# ------------------------------------------------------------
# Tabla de referencia pequeña
# ------------------------------------------------------------

payment_data = [
    (1, "Credit card"),
    (2, "Cash"),
    (3, "No charge"),
    (4, "Dispute"),
    (5, "Unknown"),
    (6, "Voided trip")
]

small_df = spark.createDataFrame(
    payment_data,
    ["payment_type", "payment_description"]
)


# ------------------------------------------------------------
# JOIN CONVENCIONAL
# ------------------------------------------------------------

original_threshold = spark.conf.get(
    "spark.sql.autoBroadcastJoinThreshold"
)

# Desactivamos el broadcast automático para poder comparar
spark.conf.set(
    "spark.sql.autoBroadcastJoinThreshold",
    -1
)

normal_join = df_file.join(
    small_df,
    on="payment_type",
    how="left"
)

print("=== PLAN — JOIN CONVENCIONAL ===")
normal_join.explain()


normal_join_time = time_it(
    lambda: (
        normal_join
        .groupBy("payment_description")
        .count()
        .collect()
    ),
    experiment="join_strategy",
    strategy="sort_merge_join"
)


# ------------------------------------------------------------
# BROADCAST JOIN
# ------------------------------------------------------------

broadcast_join = df_file.join(
    F.broadcast(small_df),
    on="payment_type",
    how="left"
)

print("\n=== PLAN — BROADCAST JOIN ===")
broadcast_join.explain()


broadcast_join_time = time_it(
    lambda: (
        broadcast_join
        .groupBy("payment_description")
        .count()
        .collect()
    ),
    experiment="join_strategy",
    strategy="broadcast_hash_join"
)


# Restauramos la configuración original
spark.conf.set(
    "spark.sql.autoBroadcastJoinThreshold",
    original_threshold
)


# ------------------------------------------------------------
# RESULTADOS
# ------------------------------------------------------------

print("\n=== RESULTADOS ===")
print(f"Join convencional: {normal_join_time:.3f} s")
print(f"Broadcast join:    {broadcast_join_time:.3f} s")

=== PLAN — JOIN CONVENCIONAL ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [payment_type#53L, VendorID#44, tpep_pickup_datetime#45, tpep_dropoff_datetime#46, passenger_count#47L, trip_distance#48, RatecodeID#49L, store_and_fwd_flag#50, PULocationID#51, DOLocationID#52, fare_amount#54, extra#55, mta_tax#56, tip_amount#57, tolls_amount#58, improvement_surcharge#59, total_amount#60, congestion_surcharge#61, Airport_fee#62, cbd_congestion_fee#63, payment_description#9418]
   +- SortMergeJoin [payment_type#53L], [payment_type#9417L], LeftOuter
      :- Sort [payment_type#53L ASC NULLS FIRST], false, 0
      :  +- Exchange hashpartitioning(payment_type#53L, 4), ENSURE_REQUIREMENTS, [plan_id=3727]
      :     +- FileScan parquet [VendorID#44,tpep_pickup_datetime#45,tpep_dropoff_datetime#46,passenger_count#47L,trip_distance#48,RatecodeID#49L,store_and_fwd_flag#50,PULocationID#51,DOLocationID#52,payment_type#53L,fare_amount#54,extra#55,mta_tax#56,tip_amount#57,tolls_amo


=== PLAN — BROADCAST JOIN ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [payment_type#53L, VendorID#44, tpep_pickup_datetime#45, tpep_dropoff_datetime#46, passenger_count#47L, trip_distance#48, RatecodeID#49L, store_and_fwd_flag#50, PULocationID#51, DOLocationID#52, fare_amount#54, extra#55, mta_tax#56, tip_amount#57, tolls_amount#58, improvement_surcharge#59, total_amount#60, congestion_surcharge#61, Airport_fee#62, cbd_congestion_fee#63, payment_description#9418]
   +- BroadcastHashJoin [payment_type#53L], [payment_type#9417L], LeftOuter, BuildRight, false, false
      :- FileScan parquet [VendorID#44,tpep_pickup_datetime#45,tpep_dropoff_datetime#46,passenger_count#47L,trip_distance#48,RatecodeID#49L,store_and_fwd_flag#50,PULocationID#51,DOLocationID#52,payment_type#53L,fare_amount#54,extra#55,mta_tax#56,tip_amount#57,tolls_amount#58,improvement_surcharge#59,total_amount#60,congestion_surcharge#61,Airport_fee#62,cbd_congestion_fee#63] Batched: true, DataFilt

In [15]:
print("=== CONFIGURACIÓN DE AUTO BROADCAST JOIN ===")

threshold_raw = spark.conf.get(
    "spark.sql.autoBroadcastJoinThreshold"
)

threshold_bytes = int(threshold_raw.rstrip("b"))
threshold_mb = threshold_bytes / (1024 ** 2)

print(f"autoBroadcastJoinThreshold: {threshold_raw}")
print(f"Umbral: {threshold_mb:.2f} MB")

=== CONFIGURACIÓN DE AUTO BROADCAST JOIN ===
autoBroadcastJoinThreshold: 10485760b
Umbral: 10.00 MB


<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### 📡 Conclusiones — Join convencional vs Broadcast Join

Se compararon dos estrategias para unir el dataset principal de **4.090.836 registros** con una pequeña tabla de referencia de **6 registros** mediante `payment_type`.

| Estrategia | Plan de ejecución | Mediana |
|---|---|---:|
| Join convencional | **SortMergeJoin** | **1,304 s** |
| Broadcast join | **BroadcastHashJoin** | **0,298 s** |

El **broadcast join fue aproximadamente 4,4 veces más rápido** en este experimento.

#### 🔀 Join convencional — SortMergeJoin

Al desactivar temporalmente el broadcast automático, Spark seleccionó un **SortMergeJoin**.

El plan físico muestra operaciones **Exchange hashpartitioning(payment_type, 4)** en ambos lados del join. Spark necesita redistribuir los datos según `payment_type` para conseguir que los registros con la misma clave puedan procesarse conjuntamente. El número de particiones del shuffle (4) es el valor de `spark.sql.shuffle.partitions` configurado en esta sesión.

Después de la redistribución, los datos se ordenan por la clave y se ejecuta el join.

Por tanto, esta estrategia implica **shuffle + ordenación**, incluyendo la redistribución de la tabla principal de millones de registros.

#### 📡 Broadcast Join — BroadcastHashJoin

Al utilizar `broadcast(small_df)`, Spark seleccionó un **BroadcastHashJoin** con `BuildRight`.

El plan contiene un **BroadcastExchange** sobre la tabla pequeña. Spark construye una estructura hash a partir de `small_df` y la distribuye para que las particiones de la tabla grande puedan realizar el join localmente.

De esta forma se evita realizar un shuffle de la tabla principal.

#### 💡 BroadcastExchange también implica movimiento de datos

Un broadcast join no significa que no exista ningún movimiento de datos.

La diferencia está en **qué datos se mueven**:

**SortMergeJoin → redistribuye los datos por la clave del join, incluyendo la tabla grande.**

**BroadcastHashJoin → distribuye la tabla pequeña y mantiene la tabla grande sin ese shuffle.**

En este experimento, distribuir una tabla de únicamente **6 registros** resulta mucho más barato que redistribuir los millones de registros del dataset principal.

#### ⚠️ ¿Cuándo utilizar Broadcast Join?

El broadcast join resulta especialmente adecuado cuando **una de las tablas es suficientemente pequeña para ser replicada en los executors**.

No debe forzarse indiscriminadamente sobre tablas grandes, ya que replicarlas puede aumentar significativamente el consumo de memoria.

Además, Spark puede seleccionar automáticamente esta estrategia cuando estima que una tabla está por debajo del umbral configurado mediante **spark.sql.autoBroadcastJoinThreshold**, que en esta sesión vale **10 MB** (10.485.760 bytes). En este experimento el broadcast automático se desactivó temporalmente para poder comparar ambas estrategias de forma controlada.

> 💡 **Conclusión:** cuando una tabla grande se une con una tabla de referencia pequeña, un BroadcastHashJoin puede evitar el costoso shuffle de la tabla grande. La mejora no procede simplemente de tener menos datos, sino de cambiar la estrategia física utilizada para ejecutar el join.

</div>

## 9. Experimento 6 — Particionado físico en escritura

El dataset se escribe dos veces, con columnas `year`/`month`/`day` derivadas de la fecha de recogida: sin `partitionBy` y con `partitionBy("year", "month", "day")`. Se compara la estructura resultante y el coste de una lectura filtrada posterior.

Antes de escribir se descartan los viajes con fecha de recogida fuera de mayo de 2026 (el dataset incluye algunos registros erróneos, como fechas de 2008 o 2009), para que el particionado refleje solo el periodo real del dataset.

In [16]:
# ============================================================
# EXPERIMENTO 6 — PARTICIONADO FÍSICO CON partitionBy
# ============================================================

from pyspark.sql import functions as F
from pathlib import Path
import time

# ------------------------------------------------------------
# 1. Preparación
# ------------------------------------------------------------

# Periodo real del dataset: mayo de 2026.
# Se descartan registros con fechas erróneas (p. ej. 2008-12-31, 2009-01-01)
# o que caen fuera del mes (30 de abril, 1 de junio).
PERIOD_START = "2026-05-01"
PERIOD_END = "2026-06-01"

df_partition_test = (
    df_file
    .filter(
        (F.col("tpep_pickup_datetime") >= F.to_timestamp(F.lit(PERIOD_START))) &
        (F.col("tpep_pickup_datetime") < F.to_timestamp(F.lit(PERIOD_END)))
    )
    .withColumn("year", F.year("tpep_pickup_datetime"))
    .withColumn("month", F.month("tpep_pickup_datetime"))
    .withColumn("day", F.dayofmonth("tpep_pickup_datetime"))
)

n_total = df_file.count()
n_kept = df_partition_test.count()

print("=== FILTRO DE FECHAS ===")
print(f"Registros originales:  {n_total:,}")
print(f"Registros en el mes:   {n_kept:,}")
print(f"Registros descartados: {n_total - n_kept:,}")

OUTPUT_BASE = Path("output/partitioning_test")
OUTPUT_NORMAL = OUTPUT_BASE / "normal"
OUTPUT_PARTITIONED = OUTPUT_BASE / "partitioned"


# ------------------------------------------------------------
# 2. Escritura SIN partitionBy
# ------------------------------------------------------------

(
    df_partition_test
    .write
    .mode("overwrite")
    .parquet(str(OUTPUT_NORMAL))
)


# ------------------------------------------------------------
# 3. Escritura CON partitionBy
# ------------------------------------------------------------

(
    df_partition_test
    .write
    .mode("overwrite")
    .partitionBy("year", "month", "day")
    .parquet(str(OUTPUT_PARTITIONED))
)


# ------------------------------------------------------------
# 4. Estadísticas de los ficheros generados
# ------------------------------------------------------------

def parquet_stats(path):
    files = list(path.rglob("*.parquet"))

    sizes = [
        file.stat().st_size / (1024 ** 2)
        for file in files
    ]

    return {
        "n_files": len(files),
        "total_mb": sum(sizes),
        "avg_mb": sum(sizes) / len(sizes) if sizes else 0
    }


normal_stats = parquet_stats(OUTPUT_NORMAL)
partitioned_stats = parquet_stats(OUTPUT_PARTITIONED)

print("=== FICHEROS GENERADOS ===")

print("\nSin partitionBy")
print(f"Nº ficheros:   {normal_stats['n_files']}")
print(f"Tamaño total:  {normal_stats['total_mb']:.2f} MB")
print(f"Tamaño medio:  {normal_stats['avg_mb']:.2f} MB")

print("\nCon partitionBy")
print(f"Nº ficheros:   {partitioned_stats['n_files']}")
print(f"Tamaño total:  {partitioned_stats['total_mb']:.2f} MB")
print(f"Tamaño medio:  {partitioned_stats['avg_mb']:.2f} MB")


# ------------------------------------------------------------
# 5. Lectura
# ------------------------------------------------------------

df_normal = spark.read.parquet(str(OUTPUT_NORMAL))
df_partitioned = spark.read.parquet(str(OUTPUT_PARTITIONED))


# Elegimos un día concreto del mes 
target_year = 2026
target_month = 5
target_day = 15

normal_filtered = df_normal.filter(
    (F.col("year") == target_year) &
    (F.col("month") == target_month) &
    (F.col("day") == target_day)
)

partitioned_filtered = df_partitioned.filter(
    (F.col("year") == target_year) &
    (F.col("month") == target_month) &
    (F.col("day") == target_day)
)

# ------------------------------------------------------------
# 6. Planes físicos
# ------------------------------------------------------------

print("\n=== PLAN — SIN partitionBy ===")
normal_filtered.explain()

print("\n=== PLAN — CON partitionBy ===")
partitioned_filtered.explain()


# ------------------------------------------------------------
# 7. Benchmark
# ------------------------------------------------------------

normal_read_time = time_it(
    lambda: normal_filtered
        .agg(F.avg("fare_amount"))
        .collect(),
    experiment="physical_partitioning",
    strategy="without_partitionBy"
)

partitioned_read_time = time_it(
    lambda: partitioned_filtered
        .agg(F.avg("fare_amount"))
        .collect(),
    experiment="physical_partitioning",
    strategy="partitionBy"
)

print("\n=== RESULTADOS LECTURA ===")
print(f"Sin partitionBy: {normal_read_time:.3f} s")
print(f"Con partitionBy:{partitioned_read_time:.3f} s")

=== FILTRO DE FECHAS ===
Registros originales:  4,090,836
Registros en el mes:   4,090,822
Registros descartados: 14


=== FICHEROS GENERADOS ===

Sin partitionBy
Nº ficheros:   4
Tamaño total:  80.36 MB
Tamaño medio:  20.09 MB

Con partitionBy
Nº ficheros:   65
Tamaño total:  81.89 MB
Tamaño medio:  1.26 MB

=== PLAN — SIN partitionBy ===
== Physical Plan ==
*(1) Filter (((((isnotnull(year#9693) AND isnotnull(month#9694)) AND isnotnull(day#9695)) AND (year#9693 = 2026)) AND (month#9694 = 5)) AND (day#9695 = 15))
+- *(1) ColumnarToRow
   +- FileScan parquet [VendorID#9673,tpep_pickup_datetime#9674,tpep_dropoff_datetime#9675,passenger_count#9676L,trip_distance#9677,RatecodeID#9678L,store_and_fwd_flag#9679,PULocationID#9680,DOLocationID#9681,payment_type#9682L,fare_amount#9683,extra#9684,mta_tax#9685,tip_amount#9686,tolls_amount#9687,improvement_surcharge#9688,total_amount#9689,congestion_surcharge#9690,Airport_fee#9691,cbd_congestion_fee#9692,year#9693,month#9694,day#9695] Batched: true, DataFilters: [isnotnull(year#9693), isnotnull(month#9694), isnotnull(day#9695), (year#9693 = 2026), (month#969..., Fo

<div style="background-color: #162B36; color: #E6EDF3; padding: 20px; border-radius: 12px; border-left: 5px solid #38BDF8;">

### 🗂️ Conclusiones — Particionado físico, Partition Pruning y Small Files

Antes de escribir se descartaron los registros con fecha de recogida fuera de mayo de 2026. Fueron solo **14 de 4.090.836** (fechas de 2008, 2009, 30 de abril y 1 de junio), pero sin este filtro cada uno habría generado su propio directorio de partición (`year=2008/month=12/...`), con ficheros de apenas unos KB.

Se comparó la escritura del mismo dataset en formato Parquet utilizando dos estrategias:

| Estrategia | Nº ficheros | Tamaño total | Tamaño medio |
|---|---:|---:|---:|
| Sin partitionBy | 4 | 80,36 MB | 20,09 MB |
| partitionBy(year, month, day) | 65 | 81,89 MB | 1,26 MB |

**Sin partitionBy** se generan 4 ficheros porque, como se vio en el baseline, solo 4 de las 8 particiones de lectura contienen datos, y cada una escribe su propio fichero.

**Con partitionBy** cada tarea de escritura genera un fichero por cada día que contiene. Los tres primeros bloques del fichero original cubren tramos consecutivos del mes (aproximadamente días 1–11, 11–20 y 20–31), pero el cuarto bloque contiene registros de los 31 días. Por eso se generan **65 ficheros para 31 días**, no 31.

Posteriormente se realizó una consulta filtrando por un día concreto:

**year = 2026, month = 5, day = 15**

> **Nota metodológica:** el dataset utilizado contiene únicamente un mes de datos, por lo que un particionado year/month aporta poca capacidad de pruning. Se añadió deliberadamente el nivel day para disponer de múltiples particiones físicas dentro del dataset y poder observar experimentalmente tanto el efecto del **partition pruning** como el coste de aumentar la granularidad del particionado. Esto no implica que year/month/day sea necesariamente el diseño adecuado para este dataset en un escenario real.

---

### 🔎 Partition Pruning

Los planes físicos muestran una diferencia clara en la forma en la que Spark aplica el filtro.

**Sin particionado físico:**

DataFilters → year = 2026, month = 5, day = 15  
PushedFilters → year = 2026, month = 5, day = 15  
PartitionFilters → vacío

Las columnas temporales están almacenadas como columnas normales dentro de los ficheros Parquet. Spark delega el filtro al lector de Parquet (PushedFilters), que puede usar las estadísticas mínimo/máximo de cada row group para saltarse bloques, pero sigue teniendo que abrir todos los ficheros y aplicar el filtro fila a fila (nodo `Filter` en el plan). No puede utilizar la estructura de directorios para descartar particiones físicas completas.

**Con partitionBy(year, month, day):**

DataFilters → vacío  
PushedFilters → vacío  
PartitionFilters → year = 2026, month = 5, day = 15

Spark reconoce year, month y day como columnas de particionado y puede utilizar estos valores para seleccionar únicamente las particiones físicas compatibles con la consulta.

El plan ya no contiene ningún nodo `Filter`: el filtrado se resuelve eligiendo qué directorios leer. Esto confirma la aplicación de **partition pruning**.

---

### ⏱️ Impacto sobre la lectura

| Lectura | Mediana |
|---|---:|
| Sin partitionBy | **0,104 s** |
| Con partitionBy | **0,047 s** |

Para la consulta diaria utilizada en el experimento, la lectura sobre el dataset particionado fue aproximadamente **2,2 veces más rápida**.

Al filtrar por un único día, Spark puede utilizar **partition pruning** para descartar las particiones correspondientes al resto de días antes de procesar sus datos.

El factor de mejora observado (**~2,2×**) es específico de este experimento y no constituye una aceleración esperable de forma general. La ganancia dependerá principalmente de la **selectividad del filtro**, del volumen de datos que pueda descartarse, de la granularidad del particionado y del entorno de ejecución.

Lo generalizable es el mecanismo observado: **si el filtro coincide con las columnas de particionado, Spark puede reducir la cantidad de datos que necesita leer mediante partition pruning**.

---

### 📁 Coste observado: Small Files

La mayor granularidad también produjo una fragmentación considerable del almacenamiento.

El número de ficheros aumentó de:

**4 → 65 ficheros**

mientras que el tamaño medio disminuyó de:

**20,09 MB → 1,26 MB**

Este resultado permite observar el **small files problem**: aumentar excesivamente la granularidad puede generar numerosos ficheros pequeños, incrementando el overhead asociado a apertura de ficheros, metadatos y planificación de tareas.

---

### ⚖️ Trade-off observado

El particionado más granular permitió observar dos efectos contrapuestos:

**Mayor granularidad**
→ filtros más selectivos  
→ mayor capacidad de partition pruning  
→ menos datos innecesarios leídos  

pero también:

**Mayor granularidad**
→ más particiones físicas  
→ potencialmente más ficheros  
→ menor tamaño medio por fichero  
→ mayor overhead de metadatos, apertura de ficheros y planificación de tareas  
→ potencial degradación del rendimiento

> 💡 **Conclusión:** el particionado diario se utilizó en este experimento como mecanismo para estudiar el comportamiento de Spark, no como recomendación de diseño para un dataset que contiene únicamente un mes. En un escenario real, las columnas y la granularidad de particionado deberían elegirse en función del volumen de datos, su distribución y, especialmente, los patrones habituales de consulta. El experimento permite comprobar el trade-off esperado: una mayor granularidad puede favorecer el partition pruning, pero también aumentar la fragmentación del almacenamiento.

</div>

## 10. Consolidar resultados

Guardar los resultados de los benchmarks en un fichero CSV para conservar las métricas obtenidas durante los experimentos y facilitar su posterior comparación y análisis.


In [17]:
from pathlib import Path
import pandas as pd

RESULTS_PATH = Path("results")
RESULTS_PATH.mkdir(parents=True, exist_ok=True)

results_df = pd.DataFrame(results)

results_df.to_csv(
    RESULTS_PATH / "experiment_results.csv",
    index=False
)

display(results_df)

,experiment,strategy,runs,median_s,partitions
0,baseline,initial_partitioning,"[0.6598509170580655, 0.22410812485031784, 0.20...",0.224108,8.0
1,partition_count,repartition,"[1.0696400001179427, 1.1094967499375343, 1.311...",1.109497,1.0
2,partition_count,repartition,"[1.6506741249468178, 1.5104540409520268, 1.510...",1.510454,4.0
3,partition_count,repartition,"[1.7986014590132982, 1.3920556670054793, 1.408...",1.408482,8.0
4,partition_count,repartition,"[2.360935000004247, 1.6120898749213666, 1.7762...",1.776241,50.0
5,partition_count,repartition,"[2.8296851669438183, 3.1635471251793206, 2.722...",2.829685,200.0
6,repartition_vs_coalesce,repartition,"[1.5179856249596924, 1.2259569170419127, 1.160...",1.225957,4.0
7,repartition_vs_coalesce,coalesce,"[0.1975699169561267, 0.15953058400191367, 0.15...",0.159531,4.0
8,cache,without_cache,"[0.6635478748939931, 0.30612912494689226, 0.27...",0.306129,NaN
9,cache,cache_materialization,[3.975602249847725],3.975602,NaN


<div style="background-color: #162B36; color: #E6EDF3; padding: 22px; border-radius: 12px; border-left: 5px solid #38BDF8;">

## 11. Conclusiones generales

Los experimentos realizados muestran que la optimización en Spark no consiste simplemente en aumentar el paralelismo o aplicar técnicas como cache, repartition o broadcast de forma sistemática. El rendimiento depende del **volumen de datos, las transformaciones realizadas, la distribución de los datos y los recursos disponibles**.

### ⚙️ Número de particiones y paralelismo

Las particiones determinan las unidades de trabajo que Spark puede procesar en paralelo.

Un número demasiado pequeño puede limitar el aprovechamiento de los recursos disponibles, mientras que un número excesivamente alto genera muchas tareas pequeñas y aumenta el overhead de planificación, coordinación y ejecución.

En este entorno, con **8 unidades de paralelismo por defecto**, aumentar progresivamente el número de particiones hasta 50 o 200 no mejoró el workload analizado. El coste adicional de gestionar y redistribuir los datos terminó superando cualquier posible beneficio de paralelismo.

Por tanto, **más particiones no implica necesariamente mayor rendimiento**. El número adecuado depende del volumen de datos, el workload y los recursos disponibles.

Además, el número de particiones no garantiza cómo se reparte el trabajo: de las **8 particiones de lectura del baseline, solo 4 contenían datos**, porque el fichero Parquet tiene 4 row groups y cada uno se lee entero en una sola partición.

Conviene distinguir además entre las particiones de entrada (las que se modifican con `repartition` o `coalesce`) y las particiones que se generan tras un shuffle, controladas por `spark.sql.shuffle.partitions` (4 en esta sesión) y ajustadas dinámicamente por AQE.

---

### 🔀 repartition frente a coalesce

Ambas operaciones permiten modificar el número de particiones, pero tienen costes diferentes.

**repartition** realiza una redistribución global de los datos mediante shuffle. Esto permite aumentar o reducir el número de particiones y obtener una distribución más equilibrada, pero introduce un coste importante de movimiento de datos.

**coalesce** está especialmente orientado a reducir el número de particiones reutilizando la distribución existente y evitando una redistribución global. Por ello puede ser considerablemente más barato, aunque no garantiza un reparto uniforme del volumen entre las particiones resultantes.

Como criterio general:

**Reducir particiones sin necesidad de redistribuir → coalesce**

**Aumentar particiones o necesitar una redistribución equilibrada → repartition**

---

### 🔄 Shuffles y Physical Plan

Los shuffles aparecen cuando Spark necesita redistribuir datos entre particiones para poder ejecutar una operación.

Operaciones como **groupBy, joins, distinct, orderBy o repartition** pueden provocar esta redistribución.

En el Physical Plan, una de las principales señales de shuffle es la aparición de operaciones **Exchange**.

Por ejemplo, el join convencional mostró operaciones **Exchange hashpartitioning** antes de ejecutar un **SortMergeJoin**, mientras que repartition introdujo un **Exchange RoundRobinPartitioning**.

Por tanto, inspeccionar el Physical Plan permite identificar dónde Spark está moviendo datos y detectar posibles puntos costosos del pipeline.

---

### 💾 Cache y persistencia

Persistir un DataFrame puede evitar recomputaciones cuando el mismo resultado intermedio se reutiliza varias veces.

Sin embargo, almacenar datos también tiene un coste.

En el experimento realizado, la cache no aportó ninguna mejora: la materialización costó **3,98 s** y, además, la reutilización fue **más lenta** que recalcular (0,52 s frente a 0,31 s). Parte del DataFrame no cabía en memoria y se leía de disco, mientras que la lectura de Parquet sin cache ya era muy barata gracias a la lectura columnar.

Además, Spark detectó presión de memoria y almacenó parte de los bloques en disco, comportamiento permitido por el StorageLevel utilizado (**Disk Memory Deserialized**, equivalente a `MEMORY_AND_DISK`).

Por tanto, cachear resulta especialmente útil cuando:

**el DataFrame es costoso de calcular + se reutiliza suficientes veces + existen recursos adecuados para persistirlo.**

Cachear indiscriminadamente puede consumir memoria sin proporcionar una mejora global.

---

### 📡 Broadcast Join

Cuando una tabla grande se une con una tabla suficientemente pequeña, Spark puede distribuir la tabla pequeña y realizar el join localmente sobre las particiones de la tabla grande.

En el experimento, el **BroadcastHashJoin** evitó el shuffle de la tabla principal que sí aparecía con **SortMergeJoin**, reduciendo considerablemente el tiempo de ejecución.

El Physical Plan permitió observar la diferencia entre **Exchange hashpartitioning** y **BroadcastExchange**.

El broadcast también implica movimiento de datos, pero mueve la relación pequeña en lugar de redistribuir la tabla grande.

Spark puede seleccionar automáticamente esta estrategia cuando estima que una relación está por debajo de **spark.sql.autoBroadcastJoinThreshold**, configurado en este entorno en **10 MB**.

Por tanto, el criterio relevante no es únicamente el número de filas, sino el **tamaño de la relación que debe distribuirse**.

---

### 🗂️ Particionado físico, pruning y small files

El número de particiones utilizadas durante el procesamiento y la organización física de los datos están relacionados, pero representan conceptos diferentes.

**repartition y coalesce** modifican las particiones utilizadas durante la ejecución.

**partitionBy** organiza físicamente los datos durante la escritura utilizando columnas concretas.

El particionado físico permite aplicar **partition pruning**: cuando una consulta filtra por columnas de particionado, Spark puede descartar particiones completas y evitar leer datos innecesarios.

Sin embargo, aumentar excesivamente la granularidad del particionado puede generar muchos ficheros pequeños.

Esto introduce el denominado **small files problem**, ya que Spark debe gestionar un mayor número de ficheros, metadatos y tareas de lectura.

Por tanto, existe un equilibrio entre:

**granularidad suficiente para permitir pruning**

y

**evitar una fragmentación excesiva del almacenamiento**.

---

### 🎯 Idea principal

La optimización de Spark consiste en buscar un equilibrio entre:

**paralelismo · movimiento de datos · tamaño de las tareas · memoria · I/O · organización física del almacenamiento**

No existe una configuración universalmente óptima.

El enfoque adecuado consiste en **medir el workload, inspeccionar el Physical Plan, identificar shuffles y cuellos de botella, aplicar una optimización y volver a medir su impacto**.

</div>